## Dependence

In [2]:
import torch
import math

import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import util

In [3]:
# file path
data_path = "data/METR-LA"
adj_data = "data/sensor_graph/adj_mx.pkl"

# training params
adj_type = 'symnadj'
batch_size = 64
nb_epochs = 120
lr = 1e-3
# encoder
hid_units = 64
em_size = 128
ft_size = 2

## Data Loading

In [4]:
# data loading
sensor_ids, sensor_id_to_ind, adj_mx = util.load_adj(adj_data, adj_type)
dataloader = util.load_dataset(data_path, batch_size, batch_size, batch_size)

adj_mx = torch.stack([torch.tensor(i) for i in adj_mx], dim=0)
adj_mx = torch.squeeze(adj_mx)

## Constractive Model

In [5]:
from engine import trainer
engine = trainer(ft_size, hid_units, adj_mx, lr)

best = 1e9
best_t = 0

for epoch in range(100):
    train_loss = []
    for iter, (x, y) in enumerate(dataloader['train_loader'].get_iterator()):
        # shape: [64, 12, 207, 2]
        trainx = torch.Tensor(x)
#         trainx = trainx.transpose(1,3)
        # trainy = torch.Tensor(y)
        loss = engine.train(trainx, epoch)
#         loss.backward()
#         self.optimizer.step()
#         train_loss = torch.cat((train_loss, torch.tensor([loss])), 0)
        train_loss.append(loss)
    
#         print(train_loss)
#         log = 'Iter: {:03d}, Train Loss: {:.4f}'
#         print(log.format(iter, train_loss[-1]),flush=True)
    
    log = 'Epoch: {:03d}, Train Loss: {:.4f}'
    m_loss = np.mean(train_loss)
#     m_loss = np.mean(train_loss.tolist())
    if m_loss < best:
        best = loss
        engine.save()
        best_t = epoch
    print(log.format(epoch, m_loss),flush=True)

/opt/homebrew/lib/python3.9/site-packages/torch/nn/functional.py:1806: UserWarning: nn.functional.sigmoid is deprecated. Use torch.sigmoid instead.
  warnings.warn("nn.functional.sigmoid is deprecated. Use torch.sigmoid instead.")


Epoch: 000, Train Loss: 0.6871
Epoch: 001, Train Loss: 0.6710
Epoch: 002, Train Loss: 0.6659
Epoch: 003, Train Loss: 0.6639
Epoch: 004, Train Loss: 0.6625
Epoch: 005, Train Loss: 0.6617
Epoch: 006, Train Loss: 0.6610
Epoch: 007, Train Loss: 0.6604
Epoch: 008, Train Loss: 0.6599
Epoch: 009, Train Loss: 0.6595
Epoch: 010, Train Loss: 0.6592
Epoch: 011, Train Loss: 0.6588
Epoch: 012, Train Loss: 0.6584
Epoch: 013, Train Loss: 0.6583
Epoch: 014, Train Loss: 0.6578
Epoch: 015, Train Loss: 0.6575
Epoch: 016, Train Loss: 0.6571
Epoch: 017, Train Loss: 0.6547
Epoch: 018, Train Loss: 0.6533
Epoch: 019, Train Loss: 0.6523
Epoch: 020, Train Loss: 0.6510
Epoch: 021, Train Loss: 0.6506
Epoch: 022, Train Loss: 0.6502
Epoch: 023, Train Loss: 0.6496
Epoch: 024, Train Loss: 0.6491
Epoch: 025, Train Loss: 0.6488
Epoch: 026, Train Loss: 0.6481
Epoch: 027, Train Loss: 0.6483
Epoch: 028, Train Loss: 0.6479
Epoch: 029, Train Loss: 0.6476
Epoch: 030, Train Loss: 0.6477
Epoch: 031, Train Loss: 0.6474
Epoch: 0

## Seq2seq

In [43]:
from models.regressor import *
import util

In [44]:
INPUT_DIM = 66
OUTPUT_DIM = 2
EMB_DIM = 128
HID_DIM = 64
N_LAYERS = 1
ENC_DROPOUT = 0.5
DEC_DROPOUT = 0.5
output_length = 12
seq_length = 12
n_features = 2

In [45]:
enc = Encoder(INPUT_DIM, EMB_DIM, HID_DIM, N_LAYERS, ENC_DROPOUT)
dec = Decoder(OUTPUT_DIM, EMB_DIM, HID_DIM, N_LAYERS, DEC_DROPOUT)

In [46]:
def init_weights(m):
    for name, param in m.named_parameters():
        nn.init.uniform_(param.data, -0.08, 0.08)

In [47]:
regressor = Seq2Seq(enc, dec)
regressor.apply(init_weights)

Seq2Seq(
  (encoder): Encoder(
    (embedding): Linear(in_features=66, out_features=128, bias=True)
    (rnn): LSTM(128, 64)
  )
  (decoder): Decoder(
    (embedding): Linear(in_features=2, out_features=128, bias=True)
    (rnn): LSTM(128, 64)
    (fc_out): Linear(in_features=64, out_features=2, bias=True)
  )
)

In [48]:
from models import STDGI 
# print('Loading {}th epoch'.format(best_t))
model = STDGI(n_features, HID_DIM)
model.load_state_dict(torch.load('best_dgi.pkl'))
scaler = dataloader['scaler']

optimizer1 = torch.optim.Adam(regressor.parameters(), lr=1e-3)
optimizer2 = torch.optim.Adam(regressor.parameters(), lr=1e-3 * 0.9)
optimizer3 = torch.optim.Adam(regressor.parameters(), lr=1e-3 * 0.8)
optimizer4 = torch.optim.Adam(regressor.parameters(), lr=1e-3 * 0.7)

In [ ]:
for epoch in range(100):
    regressor.train()
    optimizer = None
    if epoch < 20:
        optimizer = optimizer1
    elif epoch < 50:
        optimizer = optimizer2
    elif epoch < 80:
        optimizer = optimizer3
    else:
        optimizer = optimizer4

    train_losses = []
    for iter, (x, y) in enumerate(dataloader['train_loader'].get_iterator()):        
        trainx = torch.Tensor(x)
        trainy = torch.Tensor(y)
        trainx = trainx.transpose(0,1)
        trainy = trainy.transpose(0,1)

        src = torch.stack([model.embed(feature, adj_mx) for feature in trainx], dim=0)
        trainx = torch.cat((trainx, src), 3)
#         print(trainx.shape)
#         src = torch.stack([feature for feature in trainx], dim=0)
#         trg = torch.stack([feature for feature in trainy], dim=0)
        
        #for each node
        for i in range(trainx.size()[2]):
            optimizer.zero_grad()
            seq_inp = trainx[:, :, i, :]
            seq_true = trainy[:, :, i, :]
            seq_inp = seq_inp.squeeze()
            seq_true = seq_true.squeeze()
            
#             lastx = torch.unsqueeze(trainx[11, :, i, :], dim=1)
        
            seq_pred = regressor(seq_inp,seq_true)
            output_dim = seq_pred.shape[-1]

#             seq_pred = scaler.inverse_transform(seq_pred)
#             print(seq_pred)
#             print(seq_true)
            
            loss = util.masked_mae(seq_true, seq_pred)
            
            train_losses.append(loss.item())
            loss.backward()
            optimizer.step()
    
            train_loss1 = np.mean(train_losses)
#             if i % 200 == 0:
#                 print(f'Epoch {epoch} Node {i}: train loss {train_loss1}')
            
    #get predict number        
    train_loss = np.mean(train_losses)
    print(f'Epoch {epoch}: train loss {train_loss}')

Epoch 0: train loss 6.805905509954873
Epoch 1: train loss 6.147477217024651
Epoch 2: train loss 6.121571048555359
Epoch 3: train loss 5.9882091853284605
Epoch 4: train loss 8.567115556938061
Epoch 5: train loss 7.251772901553463
Epoch 6: train loss 28.482124321189673
Epoch 7: train loss 23.46453377554766
Epoch 8: train loss 10.30772042059476
Epoch 9: train loss 16.26988799078568
Epoch 10: train loss 11.662323956306814
Epoch 11: train loss 8.675241801730293
Epoch 12: train loss 6.717729265357369
Epoch 13: train loss 8.095098537991781
Epoch 14: train loss 6.682917099763806
Epoch 15: train loss 6.767087652406063
Epoch 16: train loss 11.818478772545783
Epoch 17: train loss 10.918361396347267
Epoch 18: train loss 8.547315313643303
Epoch 19: train loss 5.872286687301169
Epoch 20: train loss 6.739494173360139
Epoch 21: train loss 6.288574216999294
Epoch 22: train loss 6.5180638947233485
Epoch 23: train loss 7.849431375902824
Epoch 24: train loss 6.167094060102331
Epoch 25: train loss 7.603538

In [12]:
for epoch in range(100):
    regressor.train()
    optimizer = None
    if epoch < 20:
        optimizer = optimizer1
    elif epoch < 50:
        optimizer = optimizer2
    elif epoch < 80:
        optimizer = optimizer3
    else:
        optimizer = optimizer4

    train_losses = []
    for iter, (x, y) in enumerate(dataloader['train_loader'].get_iterator()):        
        trainx = torch.Tensor(x)
        trainy = torch.Tensor(y)
        trainx = trainx.transpose(0,1)
        trainy = trainy.transpose(0,1)

        src = torch.stack([model.embed(feature, adj_mx) for feature in trainx], dim=0)
        trainx = torch.cat((trainx, src), 3)
#         print(trainx.shape)
#         src = torch.stack([feature for feature in trainx], dim=0)
#         trg = torch.stack([feature for feature in trainy], dim=0)
        
        #for each node
        for i in range(trainx.size()[2]):
            optimizer.zero_grad()
            seq_inp = trainx[:, :, i, :]
            seq_true = trainy[:, :, i, :]
            seq_inp = seq_inp.squeeze()
            seq_true = seq_true.squeeze()
            
#             lastx = torch.unsqueeze(trainx[11, :, i, :], dim=1)
        
            seq_pred = regressor(seq_inp,seq_true)
            output_dim = seq_pred.shape[-1]

#             seq_pred = scaler.inverse_transform(seq_pred)
#             print(seq_pred)
#             print(seq_true)
            
            loss = util.masked_mae(seq_true, seq_pred)
            
            train_losses.append(loss.item())
            loss.backward()
            optimizer.step()
    
            train_loss1 = np.mean(train_losses)
            if i % 200 == 0:
                print(f'Epoch {epoch} Node {i}: train loss {train_loss1}')
            
    #get predict number        
    train_loss = np.mean(train_losses)
    print(f'Epoch {epoch}: train loss {train_loss}')

Epoch 0 Node 0: train loss 31.64827537536621
Epoch 0 Node 200: train loss 25.674404651964483
Epoch 0 Node 0: train loss 25.588922743613903
Epoch 0 Node 200: train loss 21.15862617948476
Epoch 0 Node 0: train loss 21.09159614551498
Epoch 0 Node 200: train loss 18.545777542804316
Epoch 0 Node 0: train loss 18.49315316903246
Epoch 0 Node 200: train loss 16.35418671086757
Epoch 0 Node 0: train loss 16.29758247758281
Epoch 0 Node 200: train loss 14.316436375151918
Epoch 0 Node 0: train loss 14.252947502686826
Epoch 0 Node 200: train loss 12.770808970187957
Epoch 0 Node 0: train loss 12.723378533726724
Epoch 0 Node 200: train loss 11.656535913701777
Epoch 0 Node 0: train loss 11.625634477148797
Epoch 0 Node 200: train loss 10.86192293811025
Epoch 0 Node 0: train loss 10.837679824495646
Epoch 0 Node 200: train loss 10.168547763265007
Epoch 0 Node 0: train loss 10.146198121974717
Epoch 0 Node 200: train loss 9.546831597025369
Epoch 0 Node 0: train loss 9.52870043501337
Epoch 0 Node 200: train 

Epoch 0 Node 0: train loss 5.11252458403259
Epoch 0 Node 200: train loss 5.100011480568718
Epoch 0 Node 0: train loss 5.099732924874197
Epoch 0 Node 200: train loss 5.089054521071556
Epoch 0 Node 0: train loss 5.088614553994446
Epoch 0 Node 200: train loss 5.090145784812265
Epoch 0 Node 0: train loss 5.090272883011674
Epoch 0 Node 200: train loss 5.084742217449382
Epoch 0 Node 0: train loss 5.084683158121455
Epoch 0 Node 200: train loss 5.079077197480682
Epoch 0 Node 0: train loss 5.078811836859161
Epoch 0 Node 200: train loss 5.069946724216921
Epoch 0 Node 0: train loss 5.069685922784062
Epoch 0 Node 200: train loss 5.065788897673237
Epoch 0 Node 0: train loss 5.066105368980584
Epoch 0 Node 200: train loss 5.061443682631203
Epoch 0 Node 0: train loss 5.061436667527996
Epoch 0 Node 200: train loss 5.091771033788523
Epoch 0 Node 0: train loss 5.093024439978067
Epoch 0 Node 200: train loss 5.103909045191813
Epoch 0 Node 0: train loss 5.104254127514536
Epoch 0 Node 200: train loss 5.11124

Epoch 0 Node 200: train loss 4.612465788588987
Epoch 0 Node 0: train loss 4.61229118980719
Epoch 0 Node 200: train loss 4.6083724168551505
Epoch 0 Node 0: train loss 4.608250368956783
Epoch 0 Node 200: train loss 4.6057762847897825
Epoch 0 Node 0: train loss 4.605819965136056
Epoch 0 Node 200: train loss 4.605629578401584
Epoch 0 Node 0: train loss 4.605738174201304
Epoch 0 Node 200: train loss 4.605251613710875
Epoch 0 Node 0: train loss 4.605181725907945
Epoch 0 Node 200: train loss 4.605027555135448
Epoch 0 Node 0: train loss 4.604902618996058
Epoch 0 Node 200: train loss 4.602630239962335
Epoch 0 Node 0: train loss 4.602528629881231
Epoch 0 Node 200: train loss 4.604566072045976
Epoch 0 Node 0: train loss 4.604733771677695
Epoch 0 Node 200: train loss 4.6018662212075165
Epoch 0 Node 0: train loss 4.601717445456325
Epoch 0 Node 200: train loss 4.599824264918535
Epoch 0 Node 0: train loss 4.599654907685608
Epoch 0 Node 200: train loss 4.595455853025609
Epoch 0 Node 0: train loss 4.59

Epoch 0 Node 200: train loss 4.53930592041235
Epoch 0 Node 0: train loss 4.539409761921745
Epoch 0 Node 200: train loss 4.557185584485645
Epoch 0 Node 0: train loss 4.557731239361225
Epoch 0 Node 200: train loss 4.554943544880608
Epoch 0 Node 0: train loss 4.554821900751245
Epoch 0 Node 200: train loss 4.553400283436453
Epoch 0 Node 0: train loss 4.553315079895528
Epoch 0 Node 200: train loss 4.554316963608269
Epoch 0 Node 0: train loss 4.5542706482596715
Epoch 0 Node 200: train loss 4.553501335798748
Epoch 0 Node 0: train loss 4.553465787741044
Epoch 0 Node 200: train loss 4.551693165929872
Epoch 0 Node 0: train loss 4.551617915420646
Epoch 0 Node 200: train loss 4.549865816016793
Epoch 0 Node 0: train loss 4.549895690628933
Epoch 0 Node 200: train loss 4.549861742408547
Epoch 0 Node 0: train loss 4.5498501281393935
Epoch 0 Node 200: train loss 4.547136081362197
Epoch 0 Node 0: train loss 4.547069150758215
Epoch 0 Node 200: train loss 4.548162722438354
Epoch 0 Node 0: train loss 4.548

Epoch 0 Node 200: train loss 4.504700930375913
Epoch 0 Node 0: train loss 4.504693105751039
Epoch 0 Node 200: train loss 4.5036160054814145
Epoch 0 Node 0: train loss 4.503497696632108
Epoch 0 Node 200: train loss 4.501831051764172
Epoch 0 Node 0: train loss 4.50172223107094
Epoch 0 Node 200: train loss 4.499171854849289
Epoch 0 Node 0: train loss 4.498991097546781
Epoch 0 Node 200: train loss 4.496550637594598
Epoch 0 Node 0: train loss 4.49637418790738
Epoch 0 Node 200: train loss 4.495180376656783
Epoch 0 Node 0: train loss 4.4950247490202235
Epoch 0 Node 200: train loss 4.493182958980253
Epoch 0 Node 0: train loss 4.493020806750602
Epoch 0 Node 200: train loss 4.491574600138098
Epoch 0 Node 0: train loss 4.49141040955465
Epoch 0 Node 200: train loss 4.49143590005364
Epoch 0 Node 0: train loss 4.491278327983257
Epoch 0 Node 200: train loss 4.489692779416222
Epoch 0 Node 0: train loss 4.489565805999722
Epoch 0 Node 200: train loss 4.490252228262771
Epoch 0 Node 0: train loss 4.490146

...

Epoch 10 Node 200: train loss 7.075813107135122
Epoch 10 Node 0: train loss 7.0757292776496445
Epoch 10 Node 200: train loss 7.068458269173404
Epoch 10 Node 0: train loss 7.068296886236353
Epoch 10 Node 200: train loss 7.061236474253341
Epoch 10 Node 0: train loss 7.061072834393766
Epoch 10 Node 200: train loss 7.056688416376239
Epoch 10 Node 0: train loss 7.056556095143115
Epoch 10 Node 200: train loss 7.050132132409431
Epoch 10 Node 0: train loss 7.049958226068453
Epoch 10 Node 200: train loss 7.043716226760619
Epoch 10 Node 0: train loss 7.0435104520099125
Epoch 10 Node 200: train loss 7.040005013090963
Epoch 10 Node 0: train loss 7.039893787592559
Epoch 10 Node 200: train loss 7.034489244159476
Epoch 10 Node 0: train loss 7.034446849025236
Epoch 10 Node 200: train loss 7.032121882445546
Epoch 10 Node 0: train loss 7.032086170491034
Epoch 10 Node 200: train loss 7.025653394457884
Epoch 10 Node 0: train loss 7.025534010605803
Epoch 10 Node 200: train loss 7.019922693828848
Epoch 10 N

Epoch 11 Node 200: train loss 5.357659003801483
Epoch 11 Node 0: train loss 5.357143436474754
Epoch 11 Node 200: train loss 5.343197975644841
Epoch 11 Node 0: train loss 5.342797741901313
Epoch 11 Node 200: train loss 5.3357749385924675
Epoch 11 Node 0: train loss 5.335818898328424
Epoch 11 Node 200: train loss 5.337235363805346
Epoch 11 Node 0: train loss 5.3371187412721905
Epoch 11 Node 200: train loss 5.323875180974226
Epoch 11 Node 0: train loss 5.323230104422771
Epoch 11 Node 200: train loss 5.310382679193653
Epoch 11 Node 0: train loss 5.309727669300668
Epoch 11 Node 200: train loss 5.29222704587225
Epoch 11 Node 0: train loss 5.291469878660767
Epoch 11 Node 200: train loss 5.276300061344845
Epoch 11 Node 0: train loss 5.275756904136235
Epoch 11 Node 200: train loss 5.2614013043898264
Epoch 11 Node 0: train loss 5.260725062305632
Epoch 11 Node 200: train loss 5.24537008402278
Epoch 11 Node 0: train loss 5.244829996951095
Epoch 11 Node 200: train loss 5.235419252243654
Epoch 11 No

Epoch 11 Node 200: train loss 5.355484580528537
Epoch 11 Node 0: train loss 5.3557597672189585
Epoch 11 Node 200: train loss 5.3644608700856065
Epoch 11 Node 0: train loss 5.3649444961513515
Epoch 11 Node 200: train loss 5.365131684993395
Epoch 11 Node 0: train loss 5.364987875392544
Epoch 11 Node 200: train loss 5.3591500999910915
Epoch 11 Node 0: train loss 5.359003139986451
Epoch 11 Node 200: train loss 5.355104146045307
Epoch 11 Node 0: train loss 5.355149035587371
Epoch 11 Node 200: train loss 5.358174380845858
Epoch 11 Node 0: train loss 5.35867328019901
Epoch 11 Node 200: train loss 5.358100040347019
Epoch 11 Node 0: train loss 5.358148193711838
Epoch 11 Node 200: train loss 5.351889398167455
Epoch 11 Node 0: train loss 5.351660916141116
Epoch 11 Node 200: train loss 5.3449369261585025
Epoch 11 Node 0: train loss 5.344849639480911
Epoch 11 Node 200: train loss 5.3480126943712865
Epoch 11 Node 0: train loss 5.3480820623655605
Epoch 11 Node 200: train loss 5.348503613428616
Epoch 

Epoch 11 Node 200: train loss 5.684068989660827
Epoch 11 Node 0: train loss 5.684449940886786
Epoch 11 Node 200: train loss 5.687262861966037
Epoch 11 Node 0: train loss 5.687407251199395
Epoch 11 Node 200: train loss 5.7000730472237295
Epoch 11 Node 0: train loss 5.700525062579132
Epoch 11 Node 200: train loss 5.704793855702507
Epoch 11 Node 0: train loss 5.704792838365699
Epoch 11 Node 200: train loss 5.702336852177232
Epoch 11 Node 0: train loss 5.702239985238851
Epoch 11 Node 200: train loss 5.7007781936117725
Epoch 11 Node 0: train loss 5.700676313575045
Epoch 11 Node 200: train loss 5.703747725165671
Epoch 11 Node 0: train loss 5.703802261241172
Epoch 11 Node 200: train loss 5.703657716277216
Epoch 11 Node 0: train loss 5.703540163963736
Epoch 11 Node 200: train loss 5.702540779057074
Epoch 11 Node 0: train loss 5.702482954783126
Epoch 11 Node 200: train loss 5.703281540400311
Epoch 11 Node 0: train loss 5.703238252352055
Epoch 11 Node 200: train loss 5.702541912959309
Epoch 11 N

KeyboardInterrupt: 

In [49]:
best_loss = 10000.0
for epoch in range(2):
    train_losses = []
    regressor.train()
    for iter, (x, y) in enumerate(dataloader['train_loader'].get_iterator()):
        optimizer = None
        if epoch < 20:
            optimizer = optimizer1
        elif epoch < 50:
            optimizer = optimizer2
        elif epoch < 80:
            optimizer = optimizer3
        else:
            optimizer = optimizer4
        
        trainx = torch.Tensor(x)
        trainy = torch.Tensor(y)
        trainx = trainx.transpose(0,1)
        trainy = trainy.transpose(0,1)

        src = torch.stack([model.embed(feature, adj_mx) for feature in trainx], dim=0)
        trainx = torch.cat((trainx, src), 3)
        
        #for each node
        for i in range(trainx.size()[2]):
            optimizer.zero_grad()
            seq_inp = trainx[:, :, i, :]
            seq_true = trainy[:, :, i, :]
            seq_inp = seq_inp.squeeze()
            seq_true = seq_true.squeeze()
                    
            seq_pred = regressor(seq_inp,seq_true)
            output_dim = seq_pred.shape[-1]
            
            loss = util.masked_mae(seq_true, seq_pred)
            
            train_losses.append(loss.item())
            loss.backward()
            optimizer.step()
    
            train_loss1 = np.mean(train_losses)
            if i % 200 == 0:
                print(f'Epoch {epoch} Node {i}: train loss {train_loss1}')
            
    #get predict number        
    train_loss = np.mean(train_losses)
    print(f'Epoch {epoch}: train loss {train_loss}')
    
    val_losses = []
    regressor.eval()
    for iter, (x, y) in enumerate(dataloader['val_loader'].get_iterator()):
        testx = torch.Tensor(x)
        testy = torch.Tensor(y)
        testx = testx.transpose(0,1)
        testy = testy.transpose(0,1)

        src = torch.stack([model.embed(feature, adj_mx) for feature in testx], dim=0)
        testx = torch.cat((testx, src), 3)
        
        #for each node
        for i in range(testx.size()[2]):
            seq_inp = testx[:, :, i, :]
            seq_true = testy[:, :, i, :]
            seq_inp = seq_inp.squeeze()
            seq_true = seq_true.squeeze()
                    
            seq_pred = regressor(seq_inp,seq_true)
            output_dim = seq_pred.shape[-1]
            
            loss = util.masked_mae(seq_true, seq_pred)
            
            val_losses.append(loss.item())

            val_loss1 = np.mean(val_losses)
            if val_loss1 < best_loss:
                best_loss = val_loss1
                print(f'Epoch {epoch} Node {i}: validation loss {val_loss1}')
                torch.save(regressor.state_dict(), 'best_regressor.pt')

Epoch 0 Node 0: train loss 31.669511795043945
Epoch 0 Node 200: train loss 25.542285938168046
Epoch 0 Node 0: train loss 25.45701371706449
Epoch 0 Node 200: train loss 21.07502050960765
Epoch 0 Node 0: train loss 21.009471429112445
Epoch 0 Node 200: train loss 18.599155279194438
Epoch 0 Node 0: train loss 18.548440249882326
Epoch 0 Node 200: train loss 16.465877979486237
Epoch 0 Node 0: train loss 16.408149501670543
Epoch 0 Node 200: train loss 14.423604083796864
Epoch 0 Node 0: train loss 14.360197472727666
Epoch 0 Node 200: train loss 12.865917134704521
Epoch 0 Node 0: train loss 12.817312173539367
Epoch 0 Node 200: train loss 11.693250275575197
Epoch 0 Node 0: train loss 11.660728438763783
Epoch 0 Node 200: train loss 10.880159442605395
Epoch 0 Node 0: train loss 10.855390208668211
Epoch 0 Node 200: train loss 10.174623564583804
Epoch 0 Node 0: train loss 10.152550275201705
Epoch 0 Node 200: train loss 9.551465490731852
Epoch 0 Node 0: train loss 9.532900394851145
Epoch 0 Node 200: 

Epoch 0 Node 200: train loss 5.065727598831151
Epoch 0 Node 0: train loss 5.065455383427153
Epoch 0 Node 200: train loss 5.055057772745491
Epoch 0 Node 0: train loss 5.054635901229515
Epoch 0 Node 200: train loss 5.057478906634316
Epoch 0 Node 0: train loss 5.057570506888961
Epoch 0 Node 200: train loss 5.052667912693728
Epoch 0 Node 0: train loss 5.052645871140575
Epoch 0 Node 200: train loss 5.04773837094236
Epoch 0 Node 0: train loss 5.047517625519566
Epoch 0 Node 200: train loss 5.038976872531709
Epoch 0 Node 0: train loss 5.038738064416383
Epoch 0 Node 200: train loss 5.036961213505084
Epoch 0 Node 0: train loss 5.037331525019483
Epoch 0 Node 200: train loss 5.032430567765238
Epoch 0 Node 0: train loss 5.032447997663927
Epoch 0 Node 200: train loss 5.058638585932714
Epoch 0 Node 0: train loss 5.059623480350691
Epoch 0 Node 200: train loss 5.070997287364942
Epoch 0 Node 0: train loss 5.071340586695279
Epoch 0 Node 200: train loss 5.079976674521057
Epoch 0 Node 0: train loss 5.08014

Epoch 0 Node 200: train loss 4.6242893875814595
Epoch 0 Node 0: train loss 4.6241926425207485
Epoch 0 Node 200: train loss 4.621785373249341
Epoch 0 Node 0: train loss 4.621825914288993
Epoch 0 Node 200: train loss 4.621827360668955
Epoch 0 Node 0: train loss 4.62197529443968
Epoch 0 Node 200: train loss 4.621464315979689
Epoch 0 Node 0: train loss 4.6214051959079026
Epoch 0 Node 200: train loss 4.62114690745628
Epoch 0 Node 0: train loss 4.621031978980425
Epoch 0 Node 200: train loss 4.618906845447782
Epoch 0 Node 0: train loss 4.6188002369237795
Epoch 0 Node 200: train loss 4.620717837055158
Epoch 0 Node 0: train loss 4.620886958009808
Epoch 0 Node 200: train loss 4.617915194350514
Epoch 0 Node 0: train loss 4.617760953234301
Epoch 0 Node 200: train loss 4.61587263643081
Epoch 0 Node 0: train loss 4.6157055410882935
Epoch 0 Node 200: train loss 4.611571383590215
Epoch 0 Node 0: train loss 4.611482234559832
Epoch 0 Node 200: train loss 4.609235288815251
Epoch 0 Node 0: train loss 4.60

Epoch 0 Node 200: train loss 4.56708711224347
Epoch 0 Node 0: train loss 4.566962237616405
Epoch 0 Node 200: train loss 4.565529845862511
Epoch 0 Node 0: train loss 4.565443751702922
Epoch 0 Node 200: train loss 4.5663784281228095
Epoch 0 Node 0: train loss 4.566324287927081
Epoch 0 Node 200: train loss 4.5655889321165075
Epoch 0 Node 0: train loss 4.565550831351617
Epoch 0 Node 200: train loss 4.563728852037446
Epoch 0 Node 0: train loss 4.563642311010269
Epoch 0 Node 200: train loss 4.561842403476369
Epoch 0 Node 0: train loss 4.561883278684972
Epoch 0 Node 200: train loss 4.561785015057051
Epoch 0 Node 0: train loss 4.561774070370446
Epoch 0 Node 200: train loss 4.5590294901724535
Epoch 0 Node 0: train loss 4.558948570988257
Epoch 0 Node 200: train loss 4.560052799382388
Epoch 0 Node 0: train loss 4.5600070068082035
Epoch 0 Node 200: train loss 4.557415262514864
Epoch 0 Node 0: train loss 4.5573425235862555
Epoch 0 Node 200: train loss 4.556020193768404
Epoch 0 Node 0: train loss 4.

Epoch 0 Node 200: train loss 4.511541059133511
Epoch 0 Node 0: train loss 4.511432171867266
Epoch 0 Node 200: train loss 4.508854141021407
Epoch 0 Node 0: train loss 4.508674213469071
Epoch 0 Node 200: train loss 4.5062464667983235
Epoch 0 Node 0: train loss 4.506078432509758
Epoch 0 Node 200: train loss 4.504889647780395
Epoch 0 Node 0: train loss 4.504733256454945
Epoch 0 Node 200: train loss 4.5028481989151485
Epoch 0 Node 0: train loss 4.502676754707208
Epoch 0 Node 200: train loss 4.5012036920102
Epoch 0 Node 0: train loss 4.5010391340057385
Epoch 0 Node 200: train loss 4.501152409673097
Epoch 0 Node 0: train loss 4.501017942618734
Epoch 0 Node 200: train loss 4.49966648636799
Epoch 0 Node 0: train loss 4.499537197765313
Epoch 0 Node 200: train loss 4.500201524326887
Epoch 0 Node 0: train loss 4.500101722942622
Epoch 0 Node 200: train loss 4.498428108344975
Epoch 0 Node 0: train loss 4.498265242106285
Epoch 0 Node 200: train loss 4.497131596864813
Epoch 0 Node 0: train loss 4.4969

Epoch 1 Node 200: train loss 4.4123507934755954
Epoch 1 Node 0: train loss 4.41222036813049
Epoch 1 Node 200: train loss 4.410335440038049
Epoch 1 Node 0: train loss 4.410521312353826
Epoch 1 Node 200: train loss 4.413545523920483
Epoch 1 Node 0: train loss 4.413617202141131
Epoch 1 Node 200: train loss 4.405530227774606
Epoch 1 Node 0: train loss 4.405171803191237
Epoch 1 Node 200: train loss 4.398119848723296
Epoch 1 Node 0: train loss 4.397729468517884
Epoch 1 Node 200: train loss 4.386906449689935
Epoch 1 Node 0: train loss 4.386450047635367
Epoch 1 Node 200: train loss 4.377003977912367
Epoch 1 Node 0: train loss 4.376693813213631
Epoch 1 Node 200: train loss 4.3668487046599855
Epoch 1 Node 0: train loss 4.3664186003084655
Epoch 1 Node 200: train loss 4.357576776688855
Epoch 1 Node 0: train loss 4.35730144988238
Epoch 1 Node 200: train loss 4.3537388387279305
Epoch 1 Node 0: train loss 4.353534935285531
Epoch 1 Node 200: train loss 4.3547513563345355
Epoch 1 Node 0: train loss 4.3

Epoch 1 Node 200: train loss 4.228318181308109
Epoch 1 Node 0: train loss 4.228094707502615
Epoch 1 Node 200: train loss 4.225123426475087
Epoch 1 Node 0: train loss 4.225052606832773
Epoch 1 Node 200: train loss 4.2233715678567565
Epoch 1 Node 0: train loss 4.223511405168797
Epoch 1 Node 200: train loss 4.225032605700219
Epoch 1 Node 0: train loss 4.225396896814568
Epoch 1 Node 200: train loss 4.224670617248278
Epoch 1 Node 0: train loss 4.22474284659441
Epoch 1 Node 200: train loss 4.2217567988435025
Epoch 1 Node 0: train loss 4.2216797615591375
Epoch 1 Node 200: train loss 4.217616923533001
Epoch 1 Node 0: train loss 4.21761335496876
Epoch 1 Node 200: train loss 4.221156246608371
Epoch 1 Node 0: train loss 4.221305562603986
Epoch 1 Node 200: train loss 4.221231558104035
Epoch 1 Node 0: train loss 4.221313660782133
Epoch 1 Node 200: train loss 4.230641746560848
Epoch 1 Node 0: train loss 4.230966526338382
Epoch 1 Node 200: train loss 4.21438327894907
Epoch 1 Node 0: train loss 4.2137

Epoch 1 Node 200: train loss 4.267396730256679
Epoch 1 Node 0: train loss 4.267372287996622
Epoch 1 Node 200: train loss 4.265866201300603
Epoch 1 Node 0: train loss 4.26580049548622
Epoch 1 Node 200: train loss 4.26565418913932
Epoch 1 Node 0: train loss 4.265640296577229
Epoch 1 Node 200: train loss 4.268760462514466
Epoch 1 Node 0: train loss 4.268891618264012
Epoch 1 Node 200: train loss 4.270079304438417
Epoch 1 Node 0: train loss 4.269980982551583
Epoch 1 Node 200: train loss 4.269568255171039
Epoch 1 Node 0: train loss 4.2695405180985775
Epoch 1 Node 200: train loss 4.271907291314119
Epoch 1 Node 0: train loss 4.271916540853077
Epoch 1 Node 200: train loss 4.272279374823602
Epoch 1 Node 0: train loss 4.272238639588205
Epoch 1 Node 200: train loss 4.272715517797101
Epoch 1 Node 0: train loss 4.272832600857053
Epoch 1 Node 200: train loss 4.27575085103573
Epoch 1 Node 0: train loss 4.2758293205819955
Epoch 1 Node 200: train loss 4.274795333062038
Epoch 1 Node 0: train loss 4.27474

Epoch 1 Node 200: train loss 4.280256800006918
Epoch 1 Node 0: train loss 4.28028682952909
Epoch 1 Node 200: train loss 4.289172042131282
Epoch 1 Node 0: train loss 4.2893683581811315
Epoch 1 Node 200: train loss 4.288728747697376
Epoch 1 Node 0: train loss 4.288684684839938
Epoch 1 Node 200: train loss 4.2898458874491965
Epoch 1 Node 0: train loss 4.289855451341337
Epoch 1 Node 200: train loss 4.290505228367547
Epoch 1 Node 0: train loss 4.290522619721297
Epoch 1 Node 200: train loss 4.290403900950949
Epoch 1 Node 0: train loss 4.290375743532329
Epoch 1 Node 200: train loss 4.289838592622975
Epoch 1 Node 0: train loss 4.289832118152275
Epoch 1 Node 200: train loss 4.2945222993020975
Epoch 1 Node 0: train loss 4.29463797279217
Epoch 1 Node 200: train loss 4.2945226754803985
Epoch 1 Node 0: train loss 4.294560059098931
Epoch 1 Node 200: train loss 4.296800331699248
Epoch 1 Node 0: train loss 4.2968590529940585
Epoch 1 Node 200: train loss 4.295037719091419
Epoch 1 Node 0: train loss 4.2

In [50]:
enc = Encoder(INPUT_DIM, EMB_DIM, HID_DIM, N_LAYERS, ENC_DROPOUT)
dec = Decoder(OUTPUT_DIM, EMB_DIM, HID_DIM, N_LAYERS, DEC_DROPOUT)

regressor = Seq2Seq(enc, dec)
regressor.apply(init_weights)
regressor.load_state_dict(torch.load('best_regressor.pt'))
regressor.eval()

for iter, (x, y) in enumerate(dataloader['test_loader'].get_iterator()):
    test_loss = []
    testx = torch.Tensor(x)
    testy = torch.Tensor(y)
    testx = testx.transpose(0,1)
    testy = testy.transpose(0,1)

    src = torch.stack([model.embed(feature, adj_mx) for feature in testx], dim=0)
    testx = torch.cat((testx, src), 3)

    #for each node
    for i in range(testx.size()[2]):
        seq_inp = testx[:, :, i, :]
        seq_true = testy[:, :, i, :]
        seq_inp = seq_inp.squeeze()
        seq_true = seq_true.squeeze()

        seq_pred = regressor(seq_inp,seq_true)
        output_dim = seq_pred.shape[-1]

        loss = util.masked_mae(seq_true, seq_pred)

        test_loss.append(loss.item())

#         test_loss1 = np.mean(test_loss)
test_loss = np.mean(test_loss)
print(f'train loss {test_loss}')

train loss 3.644791953085701
